
# Atari Pong with PyAOgmaNeo

## Introduction

This example demonstrates training a **PyAOgmaNeo** agent to play Atari Pong directly from pixel observations. No hand-crafted features, no replay buffers, just a brain-inspired learning algorithm that learns from experience.

**Why Pong?** This classic game requires the agent to:
- Extract meaningful patterns from raw 210×160 pixel images
- Learn temporal relationships (predicting ball trajectory)
- Discover cause-and-effect (paddle actions affect outcomes)
- Optimize long-term behavior (winning requires sustained good play)

**What makes PyAOgmaNeo different?** Unlike other deep reinforcement learning methods that:
- Require GPUs and millions of training samples
- Use replay buffers storing past experiences
- Need complex stability tricks (target networks, gradient clipping, etc.)

PyAOgmaNeo uses **Sparse Predictive Hierarchies (SPH)**, a neuromorphic approach inspired by how the brain learns:
- **Fully online learning**: Updates happen every frame, no replay needed
- **Predictive**: The system learns by predicting what will happen next
- **Extremely sparse**: Only a tiny fraction of neurons are active at any moment
- **CPU-friendly**: Runs efficiently on a laptop, even a Raspberry Pi!

>  **New to neuromorphic computing?** Check out the [History of Neuromorphic Computing](../../overview/history_of_neuromorphic_computing.md) to understand the brain-inspired principles behind SPH.

## What This Example Covers

This notebook demonstrates:

1.  Training a Pong agent from scratch using SPH
2.  How SPH processes visual information through sparse encoding
3.  Online predictive reinforcement learning in action
4.  Performance metrics and training visualizations

**Computational requirements:** Training 1000 episodes takes 1-4 hours on a modern CPU (adjustable for experimentation).

**Prerequisites:** Basic Python and familiarity with NumPy. No deep learning or neuroscience background required!

## Structure

This example covers:
1. **Dependencies** - Required packages and setup
2. **Configuration** - Hyperparameters and architecture settings
3. **Environment Setup** - The Atari Pong game interface
4. **Agent Architecture** - ImageEncoder and Hierarchy components
5. **Visual Pre-processing** - Preparing images for the network
6. **Training** - The online predictive learning loop
7. **Evaluation** - Testing the trained agent
8. **Analysis** - Comparing SPH to traditional approaches



## 0. Dependencies

This example requires the following Python libraries:

### What We're Installing

| Library | Purpose | Why We Need It |
|---------|---------|----------------|
| **PyAOgmaNeo** | The SPH neural network library | This is the brain of our agent |
| **Gymnasium** | Standard RL environment interface | Provides the Pong game |
| **ALE (Arcade Learning Environment)** | Atari game emulator | Actually runs the Pong ROM |
| **OpenCV** | Image processing | Resizes and crops game frames |
| **Matplotlib** | Visualization | Plots training curves |
| **NumPy** | Numerical computing | Array operations |

### Installation Notes

- The `[atari,accept-rom-license]` suffix automatically downloads legal Atari game ROMs
- Installation takes ~2-5 minutes depending on your internet connection
- If you already have these installed, you can skip this cell

>  **More details:** See the full [Installation Guide](../../getting_started/installation.md) for troubleshooting and alternative installation methods.


In [ ]:

# install dependencies into this environment.
# Run this once.
# %pip install "gymnasium[atari,accept-rom-license]" ale_py opencv-python pyaogmaneo



## 1. Setup and Imports

This section imports required libraries and sets up the workspace.

### Import Overview

```python
import pyaogmaneo as neo  # The SPH neural network
```

This is our main library. It provides:
- `Hierarchy` - The main predictive learning network
- `ImageEncoder` - Converts pixels to sparse neural codes
- Helper functions for managing neural state

```python
import gymnasium as gym  # RL environment interface
import ale_py           # Atari emulator
```

**Gymnasium** is the standard way to interact with reinforcement learning environments. It gives us a simple API:
- `reset()` - Start a new game
- `step(action)` - Take an action, get back observation/reward
- `render()` - Visualize the game (optional)

**Why Gymnasium?** It's the maintained successor to OpenAI Gym and is now the community standard.

```python
import cv2  # OpenCV for image processing
```

This is used to resize the 210×160 game frames down to 64×64 for faster processing.


In [ ]:

import os
import time
from copy import copy
import csv

import numpy as np
import gymnasium as gym
import ale_py
import cv2

import pyaogmaneo as neo

import matplotlib.pyplot as plt

# Register Atari environments with Gymnasium
gym.register_envs(ale_py)

print("Libraries imported.")



## 2. Configuration

This section defines all hyperparameters for the agent architecture and training schedule.

### Visual Pre-Processing Parameters

```python
image_size = (64, 64, 3)      # Target (Height, Width, Channels)
crop_height_offset = 16       # Pixels to remove from top
```

**Why resize?** The raw Atari frame is 210×160×3 (RGB). That's 100,800 values per frame! By resizing to 64×64, we reduce this to 12,288 values—making learning 8× faster while keeping essential visual information.

**Why crop?** The top of the Pong screen shows the score ("15 | 12"), which:
- Changes frequently, adding noise
- Isn't needed to play well (the agent just needs to see the ball and paddles)
- Can confuse early learning

By cropping it out, we focus the agent's attention on what matters.

### SPH Architecture Parameters

```python
enc_hidden_size = (10, 10, 32)    # ImageEncoder: 10×10 grid, 32 features per position
hierarchy_size = (7, 7, 64)       # Hierarchy: 7×7 grid, 64 features per position
num_layers = 1                    # Number of hierarchy layers
```

**What's happening here?** The architecture has two stages:

1. **ImageEncoder (10×10×32):** 
   - Divides the 64×64 image into a 10×10 grid of overlapping patches
   - Each position learns 32 different possible "visual features" (edges, corners, colors, etc.)
   - At any moment, only 1 of the 32 features is active per position
   - Total capacity: 32^100 possible visual patterns (astronomically large!)

2. **Hierarchy (7×7×64):**
   - Takes the encoded visual features and builds higher-level representations
   - Each position can be in 1 of 64 states
   - Learns temporal patterns (ball movement, paddle timing)
   - Predicts both what will happen next AND what action to take

**Why these specific numbers?** They balance:
- Expressiveness (enough features to represent complex patterns)
- Sparsity (only 100 + 49 = 149 neurons active at once!)
- Computational efficiency (runs comfortably on CPU)

> **Deep dive:** Learn about [CSDR (Columnar Sparse Distributed Representations)](../../technical_guide/core_concepts.md#csdr-the-common-language) to understand this architecture's theoretical foundation.

### Training Schedule

```python
max_episodes = 1000              # Play 1000 games
max_timesteps = 10_000           # Max 10k frames per game (prevents infinite loops)
exploration_rate = 0.01          # 1% random actions (ε-greedy)
save_frequency = 50              # Checkpoint every 50 episodes
```

**1000 episodes:** That's about 1-2 million frames of experience. For comparison, DQN typically uses 10-50 million frames!

**Exploration (ε=0.01):** With 99% probability, the agent uses its learned policy. 1% of the time, it tries random actions. This helps it discover new strategies early on and avoid getting stuck in suboptimal patterns.

**Why so little exploration?** SPH learns through prediction, not just reward. Even when taking "bad" actions, the agent learns to predict their consequences, which helps it avoid them later.

### CPU Parallelism

```python
neo.set_num_threads(4)           # Use 4 CPU cores
```

PyAOgmaNeo is designed for efficient CPU execution. On a 4-core laptop, this will use ~100% CPU during training. Adjust based on your hardware:
- Laptop (4 cores): `neo.set_num_threads(4)`
- Desktop (8+ cores): `neo.set_num_threads(6)` or `(8)`
- Server: `neo.set_num_threads(12)` or more

>  **Pro tip:** Leave 1-2 cores free for your OS and other tasks to prevent system slowdown.

### Experimenting with These Values

Want to train faster? Try:
- `max_episodes = 250` (quick experiment, won't reach full performance)
- `image_size = (32, 32, 3)` (4× faster, but may hurt final quality)

Want better performance? Try:
- `hierarchy_size = (10, 10, 96)` (more capacity, slower)
- `num_layers = 2` (hierarchical abstraction, slower but learns long-term patterns)

>  **Learn more:** See the [Parameter Tuning Guide](../../technical_guide/parameter_tuning.md) for systematic hyperparameter optimization advice.


In [ ]:

# 2. Configuration

# Whether to load an existing trained model (pong.ohr / pong.oenc)
load = False

# Visual pre-processing
image_size = (64, 64, 3)     # target (H, W, C)
crop_height_offset = 16      # shift cropping window down to cut off the score area

# SPH architecture
enc_hidden_size = (10, 10, 32)   # ImageEncoder hidden size (width, height, column size)
hierarchy_size = (7, 7, 64)      # Hidden hierarchy layer size
num_layers = 1                   # Number of hierarchy layers

# Training schedule
max_episodes = 1000              # Number of training episodes
max_timesteps = 10_000           # Max steps per episode
exploration_rate = 0.01          # ε-greedy exploration probability
save_frequency = 50              # Save model every N episodes

# Parallelism for PyAOgmaNeo
neo.set_num_threads(4)

print(f"Configured for {max_episodes} training episodes.")
print(f"Image size: {image_size}, encoder: {enc_hidden_size}, hierarchy: {hierarchy_size}")



## 3. Atari Pong Environment

This section creates and configures the Gymnasium Atari Pong environment.

### What is Pong?

Pong is one of the first video games ever created (1972). It's essentially virtual table tennis:
- **Left paddle (opponent):** Controlled by a simple AI
- **Right paddle (you/agent):** This is what we'll train
- **Ball:** Bounces between paddles
- **Goal:** Hit the ball past the opponent's paddle

### The Gymnasium Interface

```python
env = gym.make("ALE/Pong-v5")
```

This creates a standard RL environment with three key methods:

**1. Reset the game:**
```python
observation, info = env.reset()
# observation: 210×160×3 RGB image of the game screen
```

**2. Take an action:**
```python
observation, reward, terminated, truncated, info = env.step(action)
# action: integer 0-5 (NOOP, FIRE, UP, DOWN, etc.)
# reward: -1 (opponent scored), 0 (rally continues), +1 (you scored!)
# terminated: True when game ends (someone reached 21 points)
```

**3. Close when done:**
```python
env.close()
```

### Understanding the Action Space

Pong has 6 possible actions:
- `0`: NOOP (do nothing)
- `1`: FIRE (start the game)
- `2`: RIGHT (not useful in Pong)
- `3`: LEFT (not useful in Pong)
- `4`: UP (move paddle up)
- `5`: DOWN (move paddle down)

In practice, the agent mainly uses actions 0, 4, and 5.

### Frame-Skipping

**Important technical detail:** The environment internally repeats each action 4 times. This means:
- When you call `step(action)`, the action is repeated for 4 game frames
- You only see every 4th frame
- This reduces the temporal resolution but makes learning faster

This is standard in Atari RL research and matches the DQN setup.

### Observation Space Details

```python
obs_shape = env.observation_space.shape  # (210, 160, 3)
```

Each observation is:
- **210 pixels tall** × **160 pixels wide**
- **3 color channels** (RGB)
- **Values 0-255** (standard 8-bit color)

The screen includes:
- Top 20 pixels: Score display ("15 | 12")
- Middle region: Playfield with paddles and ball
- Border decorations

We'll pre-process this to focus on the playfield (see next section).

### Reward Structure and Episode Length

**Reward signal:**
- `+1` when you score a point (ball passes opponent's paddle)
- `-1` when opponent scores (ball passes your paddle)
- `0` otherwise (during rallies)

**Episode termination:**
- Game ends when either player reaches **21 points**
- Maximum possible score difference: +20 to -20
- A good agent should achieve positive cumulative reward

**Typical episode length:** 800-3000 frames depending on skill levels.

> **Training insight:** Early in training, the agent will get crushed (-20 or worse per episode). Around episode 500-800, you'll start seeing positive rewards as it learns to hit the ball back consistently.

### Visualizing the Environment (Optional)

Want to see the game? Modify the environment creation:
```python
env = gym.make("ALE/Pong-v5", render_mode='human')
```

This will open a window showing the game in real-time. **Warning:** This significantly slows down training!

> **Learn more:** See [Gymnasium's Atari documentation](https://gymnasium.farama.org/environments/atari/) for details on all Atari environments.


In [ ]:

# 3. Create the Atari Pong environment

env_id = "ALE/Pong-v5"  # requires gymnasium[atari,accept-rom-license]
env = gym.make(env_id)  # add render_mode='human' to watch the game

num_actions = env.action_space.n
obs_shape = env.observation_space.shape

min_size = min(obs_shape[0], obs_shape[1])
max_size = max(obs_shape[0], obs_shape[1])

print(f"Environment: {env_id}")
print(f"Observation shape: {obs_shape}")
print(f"Number of actions: {num_actions}")
print(f"Cropping: {min_size} x {min_size} from {obs_shape[0]} x {obs_shape[1]}")



## 4. Agent Architecture

This section builds the SPH agent with an ImageEncoder for visual processing and a Hierarchy for temporal learning and action selection.

### The Two-Stage Architecture

Think of the agent's brain as having two main components working together:

```
Raw Pixels (64×64×3)  →  ImageEncoder  →  Sparse Code (10×10×32)  →  Hierarchy  →  Actions
                          [Stage 1]                                    [Stage 2]
```

### Stage 1: ImageEncoder - From Pixels to Sparse Codes

**What it does:** Converts dense pixel data into an extremely sparse neural representation.

**Input:** 64×64×3 = 12,288 numbers (pixel values 0-255)  
**Output:** 10×10 grid where each position has 1 active cell out of 32 options

**Why this transformation?**

1. **Dimensionality reduction:** 12,288 inputs → 100 active neurons (128× compression!)
2. **Sparsity:** Only 100 cells active instead of 12,288 (makes computation lightning-fast)
3. **Learned features:** The 32 options per position aren't hand-coded—they're discovered automatically during training

**How it works conceptually:**
- Imagine dividing the image into a 10×10 grid of overlapping patches
- Each patch is like a "receptive field" that a biological neuron might see
- For each patch, the encoder asks: "Which of my 32 learned features best matches what I'm seeing?"
- It picks the single best match (winner-take-all competition)

This is called a **Columnar Sparse Distributed Representation (CSDR)**—the foundational data structure in SPH.

> **Learn more:** Read about [CSDR: The Common Language](../../technical_guide/core_concepts.md#csdr-the-common-language) and [Encoders: Compression Through Reconstruction](../../technical_guide/core_concepts.md#encoders-compression-through-reconstruction).

### Stage 2: Hierarchy - From Features to Actions

**What it does:** Learns temporal patterns and predicts both future observations and optimal actions.

**Input:** The sparse code from ImageEncoder (10×10×32 CSDR) + previous action  
**Output:** Predicted next sparse code + predicted next action

**The magic of predictive learning:**

Unlike Q-learning (which asks "how much reward will I get?"), SPH asks:
- **"What will I see next?"** (predictive modeling of visual features)
- **"What action makes sense?"** (action selection)
- **"Does this action lead to good outcomes?"** (reward-modulated learning)

The hierarchy learns by trying to predict the future, then adjusting when its predictions are wrong. The reward signal (±1 when scoring) biases this learning toward actions that lead to winning.

### Two IO Streams: Vision and Action

The hierarchy has two separate input/output channels:

**IO Stream 0 (Vision):**
- Size: 10×10×32 (matches ImageEncoder output)
- Type: `neo.none` (observation only, not predicted directly)
- Purpose: Provides visual context

**IO Stream 1 (Action):**
- Size: 1×1×6 (6 possible Pong actions)
- Type: `neo.action` (reinforcement learning output)
- Purpose: Predicts which action to take

**Why separate streams?** This architecture allows the hierarchy to:
- Learn visual patterns independently of action selection
- Use visual predictions to inform action choices
- Apply reward signals specifically to the action stream

### The `importance` Parameter

```python
h.params.ios[1].importance = 0.5
```

This balances two learning objectives:
- **Predicting what happens (vision stream):** importance = 1.0 (implicit)
- **Choosing good actions (action stream):** importance = 0.5

**Why 0.5?** We want the agent to:
- Spend most of its "mental effort" understanding the visual world (higher importance)
- But also learn good actions (moderate importance)

If we set this too high (e.g., 0.9), the agent might focus too much on winning and not learn visual features well. Too low (e.g., 0.1), and it won't care enough about winning!

### Receptive Field Radii

```python
neo.IODesc(size=enc.get_hidden_size(), io_type=neo.none, up_radius=3)
neo.IODesc(size=(1, 1, num_actions), io_type=neo.action, up_radius=0, down_radius=3)
```

**`up_radius=3`:** Each position in the hierarchy looks at a 7×7 patch of encoder features (radius 3 → diameter 2×3+1=7). This lets it integrate information from a wider area.

**`down_radius=3`:** When making predictions, each position uses a 7×7 area of context. This helps generate spatially coherent predictions.

**`up_radius=0` for actions:** Actions don't need spatial structure—there's only one action to predict, so no neighboring positions needed.

> 📚 **Deep dive:** See [Receptive Fields and Connectivity](../../technical_guide/core_concepts.md#receptive-fields-and-connectivity) for the math behind these choices.

### Loading vs. Creating Models

The code checks `if load:` to either:
- **Load existing model:** Useful for resuming training or testing a pre-trained agent
- **Create from scratch:** What we'll do the first time

**Model files:**
- `pong.oenc` - ImageEncoder weights
- `pong.ohr` - Hierarchy weights

These are saved automatically during training (every 50 episodes by default).

> 📚 **Learn more:** See [State Management API](../../api_reference/state_management.md) for saving/loading models and [Hierarchy API](../../api_reference/hierarchy.md) for complete architecture documentation.


In [ ]:

# 4. Build the APong agent: ImageEncoder + Hierarchy

h = None   # Hierarchy (main predictive model)
enc = None # ImageEncoder (visual pre-encoder)

if load:
    print("Loading existing models from disk...")
    try:
        enc = neo.ImageEncoder(file_name="pong.oenc")
        h = neo.Hierarchy(file_name="pong.ohr")
        print("Models loaded.")
    except Exception as e:
        print("Failed to load models, creating new ones instead.")
        print("Reason:", repr(e))
        load = False

if not load:
    print("Creating new models from scratch...")

    # ImageEncoder: visual feature extractor (dense RGB -> sparse CSDR)
    enc = neo.ImageEncoder(
        enc_hidden_size,
        [
            neo.ImageVisibleLayerDesc(
                (image_size[1], image_size[0], image_size[2]),
                8,  # receptive field radius
            )
        ],
    )

    # Hierarchy layer descriptions
    layer_descs = []
    for _ in range(num_layers):
        ld = neo.LayerDesc()
        ld.hidden_size = hierarchy_size
        layer_descs.append(ld)

    # Two IO streams:
    #  - index 0: visual features (no direct predictions)
    #  - index 1: discrete action predictions
    h = neo.Hierarchy(
        [
            neo.IODesc(size=enc.get_hidden_size(), io_type=neo.none, up_radius=3),
            neo.IODesc(size=(1, 1, num_actions), io_type=neo.action, up_radius=0, down_radius=3),
        ],
        layer_descs,
    )

    # Balance between "regular" prediction and action learning
    h.params.ios[1].importance = 0.5

print("Models ready.")
print("  ImageEncoder hidden size:", enc.get_hidden_size())
print("  Hierarchy hidden size:", h.get_hidden_size(0))
print("  Action space size:", num_actions)



## 5. Visual Pre-Processing

This section defines the image pre-processing pipeline that crops and resizes frames for the agent.

### Why Pre-Processing Matters

Raw Atari frames are messy:
- **Too large:** 210×160 pixels = too much data to process efficiently
- **Noisy information:** The score display changes constantly but isn't useful for gameplay
- **Spatial imbalance:** The frame is rectangular (210×160), but our encoder works best with square inputs

Good pre-processing can speed up training by 5-10× and improve final performance!

### The Pre-Processing Pipeline

```
Raw Frame (210×160×3) → Crop → Resize → Clean Frame (64×64×3)
```

#### Step 1: Crop to Square

We extract a centered square region:
```python
start_row = max_size // 2 - min_size // 2 + crop_height_offset
end_row = max_size // 2 + min_size // 2 + crop_height_offset
cropped = obs[start_row:end_row, :, :]
```

**What this does:**
- Takes the middle 160×160 square from the 210×160 frame
- **Removes the top portion** containing the score
- **Adds `crop_height_offset=16`** to shift down and keep the ball/paddles centered

**Visual explanation:**
```
Original 210×160:        After cropping 160×160:
┌────────────────┐      
│  SCORE: 15|12  │  ← Remove this (top 25 pixels)
├────────────────┤       ┌────────────────┐
│                │       │                │
│    paddle |    │  →    │    paddle |    │  ← Keep this (gameplay)
│        o       │       │        o       │
│    paddle |    │       │    paddle |    │
│                │       │                │
└────────────────┘       └────────────────┘
```

#### Step 2: Resize to Network Input Size

```python
resized = cv2.resize(cropped, (64, 64), interpolation=cv2.INTER_LINEAR)
```

**What this does:**
- Shrinks the 160×160 crop down to 64×64
- Uses bilinear interpolation for smooth downsampling
- Reduces data by 6.25× (160² → 64²)

**Why 64×64?** It's a sweet spot:
- Large enough to see the ball and paddles clearly
- Small enough for fast processing
- Power of 2 (efficient for computer memory)

#### Step 3: Keep Color (Don't Convert to Grayscale)

**Important design choice:** We keep all 3 RGB channels!

Many Atari RL systems convert to grayscale to reduce data. We don't because:
- **Color helps distinguish objects:** The ball is white, paddles are white, background is black—color information helps the encoder learn these distinctions faster
- **Minimal cost:** With sparse representations, processing 3 channels vs. 1 doesn't add much computation
- **Better generalization:** Color-trained encoders transfer better to other visual tasks

### Testing the Pre-Processing

The code includes a sanity check:
```python
test_obs, _ = env.reset()
processed = preprocess_frame(test_obs)
print(f"Original frame: {test_obs.shape} -> Processed: {processed.shape}")
```

**Expected output:**
```
Original frame: (210, 160, 3) -> Processed: (64, 64, 3)
```

This confirms our pipeline is working correctly!

### Common Pre-Processing Mistakes to Avoid



> **Learn more:** See [Pre-Encoders and Pre-Decoders](../../technical_guide/core_concepts.md#pre-encoders-and-pre-decoders) for the theory behind input transformations, and [ImageEncoder API](../../api_reference/image_encoder.md) for advanced visual encoding options.


In [ ]:

def preprocess_frame(obs):
    # Preprocess a raw Atari frame for the SPH agent:
    # 1. Crop a square region, removing the score area at the top.
    # 2. Resize to the network's input resolution.
    # 3. Keep RGB channels (no grayscale conversion).

    # Crop to a centered square region and shift down to remove the score
    start_row = max_size // 2 - min_size // 2 + crop_height_offset
    end_row = max_size // 2 + min_size // 2 + crop_height_offset
    cropped = obs[start_row:end_row, :, :]

    # Resize to the network input size
    resized = cv2.resize(
        cropped,
        (image_size[0], image_size[1]),
        interpolation=cv2.INTER_LINEAR,
    )

    return resized

# Quick sanity check
test_obs, _ = env.reset()
processed = preprocess_frame(test_obs)
print(f"Original frame: {test_obs.shape} -> Processed: {processed.shape}")


In [ ]:

def train_agent(
    num_episodes=max_episodes,
    max_timesteps=max_timesteps,
    exploration_rate=exploration_rate,
    reward_scale=100.0,
    save_every=save_frequency,
):
    # Train the APong agent and log episodic rewards.
    # Returns:
    #   episode_rewards : np.ndarray
    #   ema_rewards     : np.ndarray (exponential moving average)
    #   first_win_episode : int or None (1-based index of first episode with total reward > 0)

    global h, enc

    episode_rewards = []
    ema_rewards = []
    best_reward = -np.inf
    first_win_episode = None

    print(f"Starting training for {num_episodes} episodes...\n")
    action = 0
    prev_reward = 0.0

    for episode in range(num_episodes):
        obs, info = env.reset()
        total_reward = 0.0
        prev_reward = 0.0

        for t in range(max_timesteps):
            # 1) Encode visual input
            processed_obs = preprocess_frame(obs)
            enc.step([processed_obs.ravel()], True)

            # 2) Predict next state + action, and learn from previous reward
            h.step([enc.get_hidden_cis(), [action]], True, prev_reward * reward_scale)

            # 3) Choose next action from predictions
            action = int(h.get_prediction_cis(1)[0])

            # ε-greedy exploration
            if np.random.rand() < exploration_rate:
                action = np.random.randint(0, num_actions)

            # 4) Step the environment
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            prev_reward = reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)
        if episode == 0:
            ema = total_reward
        else:
            ema = 0.9 * ema_rewards[-1] + 0.1 * total_reward
        ema_rewards.append(ema)

        if total_reward > best_reward:
            best_reward = total_reward

        if total_reward > 0 and first_win_episode is None:
            first_win_episode = episode + 1

        print(
            f"Episode {episode + 1:4d}: "
            f"{t + 1:4d} steps, "
            f"reward {total_reward:6.1f}, "
            f"EMA avg {ema:6.2f}"
        )
        if first_win_episode == episode + 1:
            print(f"  --> first positive total reward at episode {first_win_episode}!")

        if (episode + 1) % save_every == 0:
            h.save_to_file("pong.ohr")
            enc.save_to_file("pong.oenc")
            print(f"Saved intermediate models at episode {episode + 1}")

    # Save final models
    h.save_to_file("pong_final.ohr")
    enc.save_to_file("pong_final.oenc")

    # Save metrics to CSV for external tools (e.g. R)
    with open("apong_training_metrics.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["episode", "reward", "ema_reward"])
        for ep, r, ema in zip(range(1, num_episodes + 1), episode_rewards, ema_rewards):
            writer.writerow([ep, r, ema])

    print("\nTraining finished.")
    print(f"Best per-episode reward: {best_reward:.1f}")
    if first_win_episode is not None:
        print(f"First episode with positive total reward: {first_win_episode}")
    else:
        print("Agent did not achieve a positive total reward during this run.")

    return np.array(episode_rewards), np.array(ema_rewards), first_win_episode


# Run training (this may take a while for 1000 episodes)
episode_rewards, ema_rewards, first_win_episode = train_agent()


In [ ]:

episodes = np.arange(1, len(episode_rewards) + 1)

plt.figure()
plt.plot(episodes, episode_rewards, label="Episode reward")
plt.plot(episodes, ema_rewards, label="EMA reward (0.9/0.1)")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("APong training curve (PyAOgmaNeo)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

def test_agent(num_episodes=5):
    # Run the trained agent without learning and report total rewards.
    print(f"\nTesting agent for {num_episodes} episodes...\n")

    test_rewards = []
    action = 0

    for episode in range(num_episodes):
        obs, _ = env.reset()
        total_reward = 0.0

        for t in range(max_timesteps):
            processed_obs = preprocess_frame(obs)

            # Forward pass only (no learning)
            enc.step([processed_obs.ravel()], False)
            h.step([enc.get_hidden_cis(), [action]], False, 0.0)

            # Choose the predicted action
            action = int(h.get_prediction_cis(1)[0])

            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward

            if terminated or truncated:
                break

        test_rewards.append(total_reward)
        print(f"Test episode {episode + 1}: total reward {total_reward:.1f} in {t + 1} steps")

    avg_test_reward = float(np.mean(test_rewards))
    print("\nSummary over test episodes:")
    print(f"  Average reward: {avg_test_reward:.2f}")
    print(f"  Best game:      {max(test_rewards):.1f}")
    print(f"  Worst game:     {min(test_rewards):.1f}")

    return test_rewards

# Example: 3 evaluation games
test_results = test_agent(num_episodes=3)


In [ ]:

# 10. Cleanup

env.close()
print("Environment closed. Models were saved to disk if training was run.")
